# Séance 9 — Aller chercher la donnée

## 🛟 Notebook « point de reprise »

**À quoi sert ce notebook ?** Si tu as manqué la séance précédente, si ton code
ne marche pas, ou si tu t'es perdu·e en route : **ouvre celui-ci**. Le code de
départ est déjà écrit et fonctionne. Tu n'as jamais besoin d'avoir réussi
l'exercice d'avant pour suivre celui d'aujourd'hui.

**Comment l'utiliser ?**
1. Exécute les cellules du haut sans les modifier (elles remettent tout en place).
2. Descends jusqu'aux cellules `# ✏️ À TOI DE JOUER`.
3. Écris ton code à la place des `...`.

**Raccourci** : `Maj + Entrée` exécute une cellule.

---


> ## ⚠️ Avant toute chose — les 5 règles
> 1. **API d'abord**, scraping seulement si nécessaire.
> 2. Lire `robots.txt` et les CGU **avant** d'écrire la première ligne.
> 3. **Un délai entre chaque requête** (1 seconde suffit).
> 4. **Aucune donnée personnelle.**
> 5. S'identifier honnêtement dans le `User-Agent`.
>
> On s'entraîne sur `books.toscrape.com`, un site **conçu pour ça**.
> Jamais sur un site réel en séance.


In [ ]:
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

EN_TETES = {"User-Agent": "FormationPython/1.0 (exercice pedagogique)"}


## 1. Une requête HTTP, et la lecture du code de statut

In [ ]:
reponse = requests.get("https://books.toscrape.com/", headers=EN_TETES, timeout=10)

print("Statut :", reponse.status_code)   # 200 = OK, 404 = absent, 429 = trop vite
print("Type   :", reponse.headers.get("Content-Type"))
print(reponse.text[:300])


## 2. BeautifulSoup : du HTML aux données

In [ ]:
soup = BeautifulSoup(reponse.text, "html.parser")   # toujours préciser le parseur

premiere = soup.select_one("article.product_pod")
print(premiere.select_one("h3 a")["title"])
print(premiere.select_one("p.price_color").text.strip())


### Les 5 sélecteurs CSS qui couvrent 95 % des besoins

| Sélecteur | Signification |
|---|---|
| `h2` | toutes les balises `h2` |
| `.prix` | les éléments de classe `prix` |
| `#resultats` | l'élément d'identifiant `resultats` |
| `article .titre` | les `.titre` **à l'intérieur** d'un `article` |
| `a[href]` | les liens **possédant** un attribut `href` |

**Méthode** : clic droit → Inspecter → clic droit sur l'élément → Copier le sélecteur.


## 3. ⚠️ Toujours vérifier l'absence avant d'accéder

In [ ]:
def extraire_livres(html: str) -> list[dict]:
    soup = BeautifulSoup(html, "html.parser")
    livres = []

    for fiche in soup.select("article.product_pod"):
        lien = fiche.select_one("h3 a")
        prix = fiche.select_one("p.price_color")
        note = fiche.select_one("p.star-rating")

        livres.append({
            # Une seule fiche mal formée ne doit pas faire tomber le scraper.
            "titre": lien["title"] if lien else None,
            "prix": prix.text.strip() if prix else None,
            "note": note["class"][1] if note and len(note["class"]) > 1 else None,
            "url": lien["href"] if lien else None,
        })
    return livres


print(len(extraire_livres(reponse.text)), "livres sur la page 1")


---
# ✏️ À TOI DE JOUER — Le scraper paginé

Complète la fonction : parcours les pages jusqu'à un 404 ou la limite,
avec un délai entre chaque requête, et rends un DataFrame.

Corrigé complet : `fil-rouge/v6-scraper/scraper.py`.


In [ ]:
# ✏️ À TOI DE JOUER
BASE = "https://books.toscrape.com/catalogue/page-{}.html"
DELAI = 1.0   # ⚠️ ne retire JAMAIS cette ligne


def recuperer_page(numero: int) -> str | None:
    reponse = requests.get(BASE.format(numero), headers=EN_TETES, timeout=10)
    if reponse.status_code == 404:
        return None
    reponse.raise_for_status()
    reponse.encoding = reponse.apparent_encoding
    return reponse.text


def scraper(max_pages: int = 5) -> pd.DataFrame:
    tout = []
    for numero in range(1, max_pages + 1):
        html = ...
        if html is None:
            break
        tout.extend(...)
        time.sleep(DELAI)
    return pd.DataFrame(tout)


# df = scraper(3)
# df.head()
